In [ ]:
import numpy as np
import os
import json
import time
import warnings
import matplotlib.pyplot as plt
from joblib import dump, load, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.exceptions import ConvergenceWarning
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('advanced_pipeline_optimizer')

# Suppress some warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Directory configuration
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

def load_data(models_dir='../models'):
    """
    Load the preprocessed EEG data
    """
    logger.info("Loading preprocessed data...")
    
    # Find the preprocessed files
    x_files = [f for f in os.listdir(models_dir) if f.startswith('X_preprocessed') and f.endswith('.npy')]
    y_files = [f for f in os.listdir(models_dir) if f.startswith('y_labels') and f.endswith('.npy')]
    
    if not x_files or not y_files:
        raise FileNotFoundError(f"Preprocessed data files not found in {models_dir}")
    
    # Sort by timestamp to get the latest
    x_files.sort(reverse=True)
    y_files.sort(reverse=True)
    
    X = np.load(os.path.join(models_dir, x_files[0]))
    y = np.load(os.path.join(models_dir, y_files[0]))
    
    logger.info(f"Loaded data: X shape {X.shape}, y shape {y.shape}")
    logger.info(f"Classes: {np.unique(y)}")
    
    # Try to load preprocessing info
    preprocessing_info = None
    info_files = [f for f in os.listdir(models_dir) if f.startswith('preprocessing_info') and f.endswith('.json')]
    if info_files:
        info_files.sort(reverse=True)
        with open(os.path.join(models_dir, info_files[0]), 'r') as f:
            preprocessing_info = json.load(f)
    
    return X, y, preprocessing_info

def augment_data(X, y, random_state=42):
    """
    Apply SMOTE to balance classes and augment the dataset
    """
    logger.info("Applying SMOTE to balance classes...")
    try:
        smote = SMOTE(random_state=random_state)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        logger.info(f"Data after SMOTE: X shape {X_resampled.shape}, y shape {y_resampled.shape}")
        logger.info(f"Class distribution after SMOTE: {np.bincount(y_resampled)}")
        return X_resampled, y_resampled
    except Exception as e:
        logger.warning(f"SMOTE failed: {e}. Continuing with original data.")
        return X, y

def create_advanced_pipelines():
    """
    Create a variety of advanced pipelines for EEG classification
    """
    # Scalers
    scalers = {
        'standard': StandardScaler(),
        'robust': RobustScaler(),
        'quantile': QuantileTransformer(output_distribution='normal')
    }
    
    # Feature selection and dimensionality reduction
    dim_reduction = {
        'pca': PCA(n_components=0.95),
        'pca_fixed': PCA(n_components=15),  # Fixed number of components
        'ica': FastICA(n_components=15, random_state=42),
        'select_k': SelectKBest(f_classif, k=20),
        'select_k_mi': SelectKBest(mutual_info_classif, k=20),
        'lda': LDA(n_components=2)  # LDA for dimensionality reduction
    }
    
    # Classifiers
    classifiers = {
        'svm_rbf': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
        'svm_linear': SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42),
        'rf': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        'gb': GradientBoostingClassifier(n_estimators=100, random_state=42),
        'lr': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'knn': KNeighborsClassifier(n_neighbors=5),
        'mlp': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
    }
    
    # Create pipelines for each combination
    pipelines = {}
    
    for scaler_name, scaler in scalers.items():
        for dim_red_name, dim_red in dim_reduction.items():
            for clf_name, clf in classifiers.items():
                # Skip some combinations that might not work well
                if dim_red_name == 'lda' and clf_name in ['lda', 'qda']:
                    continue
                
                pipeline_name = f"{scaler_name}_{dim_red_name}_{clf_name}"
                
                pipeline = Pipeline([
                    ('scaler', scaler),
                    ('dim_reduction', dim_red),
                    ('classifier', clf)
                ])
                
                pipelines[pipeline_name] = pipeline
    
    # Add SMOTE pipelines for imbalanced data
    smote_pipelines = {}
    
    for pipeline_name, pipeline in list(pipelines.items())[:5]:  # Add SMOTE to some of the best pipelines
        smote_pipeline = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('pipeline', pipeline)
        ])
        smote_pipelines[f"smote_{pipeline_name}"] = smote_pipeline
    
    pipelines.update(smote_pipelines)
    
    # Add ensemble models
    ensemble_pipelines = {}
    
    # Voting classifier with best estimators
    voting_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('dim_reduction', PCA(n_components=0.95)),
        ('voting', VotingClassifier(
            estimators=[
                ('svm', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)),
                ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
                ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
            ],
            voting='soft'
        ))
    ])
    
    ensemble_pipelines['voting_ensemble'] = voting_pipeline
    
    pipelines.update(ensemble_pipelines)
    
    # Add custom pipelines with specific hyperparameters
    custom_pipelines = {
        'tuned_svm': Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95)),
            ('classifier', SVC(C=10, gamma='scale', kernel='rbf', probability=True, class_weight='balanced', random_state=42))
        ]),
        'tuned_rf': Pipeline([
            ('scaler', RobustScaler()),
            ('pca', PCA(n_components=20)),
            ('classifier', RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_split=2, min_samples_leaf=1, class_weight='balanced', random_state=42))
        ])
    }
    
    pipelines.update(custom_pipelines)
    
    logger.info(f"Created {len(pipelines)} pipelines for evaluation")
    return pipelines

def evaluate_pipeline(pipeline, pipeline_name, X, y, cv, scoring='accuracy'):
    """
    Evaluate a single pipeline with cross-validation
    """
    try:
        start_time = time.time()
        scores = cross_val_score(
            pipeline, X, y, 
            cv=cv, 
            scoring=scoring,
            n_jobs=1,  # Running in parallel at higher level
            error_score='raise'
        )
        
        eval_time = time.time() - start_time
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        logger.info(f"{pipeline_name}: {scoring} = {mean_score:.4f} ± {std_score:.4f} (Time: {eval_time:.2f}s)")
        
        return {
            'name': pipeline_name,
            'accuracy': float(mean_score),
            'std': float(std_score),
            'time': float(eval_time),
            'pipeline': pipeline
        }
    except Exception as e:
        logger.error(f"Error evaluating {pipeline_name}: {str(e)}")
        return None

def optimize_hyperparameters(pipeline, X, y, cv):
    """
    Optimize hyperparameters for the top pipeline
    """
    logger.info(f"Optimizing hyperparameters for the top pipeline...")
    
    # Identify the pipeline type and create appropriate param grid
    if hasattr(pipeline, 'named_steps'):
        steps = pipeline.named_steps
    elif hasattr(pipeline, 'steps'):
        steps = dict(pipeline.steps)
    else:
        logger.warning("Pipeline structure not recognized, skipping hyperparameter optimization")
        return pipeline
    
    param_grid = {}
    
    # Check if it's a SMOTE pipeline
    is_smote = False
    if hasattr(pipeline, 'steps') and pipeline.steps[0][0] == 'smote':
        is_smote = True
        if hasattr(pipeline, 'steps') and len(pipeline.steps) > 1 and hasattr(pipeline.steps[1][1], 'named_steps'):
            steps = pipeline.steps[1][1].named_steps
    
    # Add parameters based on classifier type
    if 'classifier' in steps:
        clf = steps['classifier']
        if isinstance(clf, SVC):
            param_prefix = 'pipeline__' if is_smote else ''
            param_grid.update({
                f'{param_prefix}classifier__C': [0.1, 1, 10, 100],
                f'{param_prefix}classifier__gamma': ['scale', 'auto', 0.01, 0.1]
            })
        elif isinstance(clf, RandomForestClassifier):
            param_prefix = 'pipeline__' if is_smote else ''
            param_grid.update({
                f'{param_prefix}classifier__n_estimators': [100, 200, 300],
                f'{param_prefix}classifier__max_depth': [None, 10, 20, 30],
                f'{param_prefix}classifier__min_samples_split': [2, 5, 10]
            })
        elif isinstance(clf, GradientBoostingClassifier):
            param_prefix = 'pipeline__' if is_smote else ''
            param_grid.update({
                f'{param_prefix}classifier__n_estimators': [50, 100, 200],
                f'{param_prefix}classifier__learning_rate': [0.01, 0.1, 0.2],
                f'{param_prefix}classifier__max_depth': [3, 5, 7]
            })
        elif isinstance(clf, MLPClassifier):
            param_prefix = 'pipeline__' if is_smote else ''
            param_grid.update({
                f'{param_prefix}classifier__hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
                f'{param_prefix}classifier__alpha': [0.0001, 0.001, 0.01],
                f'{param_prefix}classifier__learning_rate': ['constant', 'adaptive']
            })
    
    # Add PCA parameters if present
    if 'dim_reduction' in steps and isinstance(steps['dim_reduction'], PCA):
        param_prefix = 'pipeline__' if is_smote else ''
        param_grid.update({
            f'{param_prefix}dim_reduction__n_components': [0.85, 0.9, 0.95, 0.99]
        })
    
    # If no parameters to optimize, return original pipeline
    if not param_grid:
        logger.warning("No parameters to optimize, returning original pipeline")
        return pipeline
    
    # Use F1 score for optimization
    f1_scorer = make_scorer(f1_score, average='weighted')
    
    # Create and run GridSearchCV
    logger.info(f"Running grid search with parameters: {param_grid}")
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        scoring=f1_scorer,
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    try:
        grid_search.fit(X, y)
        logger.info(f"Best parameters: {grid_search.best_params_}")
        logger.info(f"Best score: {grid_search.best_score_:.4f}")
        return grid_search.best_estimator_
    except Exception as e:
        logger.error(f"Error during hyperparameter optimization: {e}")
        return pipeline

def evaluate_and_find_best_pipeline(X, y, scoring='accuracy'):
    """
    Evaluate all pipelines and return the best one
    """
    logger.info(f"Starting advanced pipeline evaluation for best {scoring}...")
    
    # Create pipelines
    pipelines = create_advanced_pipelines()
    
    # Create cross-validation object
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Evaluate pipelines in parallel
    logger.info(f"Evaluating {len(pipelines)} pipelines with cross-validation...")
    
    results = Parallel(n_jobs=-1)(
        delayed(evaluate_pipeline)(pipeline, name, X, y, cv, scoring)
        for name, pipeline in pipelines.items()
    )
    
    # Filter out failed evaluations
    results = [r for r in results if r is not None]
    
    # Sort by score
    results.sort(key=lambda x: x['accuracy'], reverse=True)
    
    # Show top 5 pipelines
    logger.info("\nTop 5 pipelines:")
    for i, result in enumerate(results[:5]):
        logger.info(f"{i+1}. {result['name']}: {result['accuracy']:.4f} ± {result['std']:.4f} (Time: {result['time']:.2f}s)")
    
    if not results:
        logger.error("All pipeline evaluations failed")
        return None, None
    
    # Get best pipeline
    best_result = results[0]
    best_pipeline_name = best_result['name']
    best_pipeline = best_result['pipeline']
    best_accuracy = best_result['accuracy']
    
    logger.info(f"\nBest pipeline: {best_pipeline_name} with {scoring}: {best_accuracy:.4f}")
    
    # Try to optimize the best pipeline
    optimized_pipeline = optimize_hyperparameters(best_pipeline, X, y, cv)
    
    # Evaluate optimized pipeline
    logger.info("Evaluating optimized pipeline...")
    optimized_scores = cross_val_score(
        optimized_pipeline, X, y, 
        cv=cv, 
        scoring=scoring,
        n_jobs=-1
    )
    
    optimized_accuracy = np.mean(optimized_scores)
    logger.info(f"Optimized pipeline {scoring}: {optimized_accuracy:.4f} ± {np.std(optimized_scores):.4f}")
    
    # Use optimized if better, otherwise stick with original
    if optimized_accuracy > best_accuracy:
        logger.info("Using optimized pipeline (better performance)")
        best_pipeline = optimized_pipeline
        best_accuracy = optimized_accuracy
        best_pipeline_name = f"{best_pipeline_name}_optimized"
    else:
        logger.info("Keeping original pipeline (better performance)")
    
    # Train final pipeline on all data
    logger.info("Training final pipeline on all data...")
    start_time = time.time()
    best_pipeline.fit(X, y)
    training_time = time.time() - start_time
    
    # Save the best pipeline
    pipeline_path = os.path.join(models_dir, 'best_pipeline.joblib')
    dump(best_pipeline, pipeline_path)
    
    # Save pipeline info
    pipeline_info = {
        'best_pipeline_name': best_pipeline_name,
        'best_accuracy': float(best_accuracy),
        'training_time': float(training_time),
        'pipeline_path': pipeline_path,
        'evaluation_results': [
            {
                'name': r['name'],
                'accuracy': r['accuracy'],
                'std': r['std'],
                'time': r['time']
            }
            for r in results[:10]  # Save top 10 results
        ]
    }
    
    info_path = os.path.join(models_dir, 'best_pipeline_info.json')
    with open(info_path, 'w') as f:
        json.dump(pipeline_info, f, indent=4)
    
    logger.info(f"Best pipeline saved to: {pipeline_path}")
    logger.info(f"Pipeline info saved to: {info_path}")
    
    # Visualize results
    plt.figure(figsize=(12, 8))
    
    # Get top 10 pipelines for visualization
    top_results = results[:10]
    
    names = [r['name'] for r in top_results]
    accuracies = [r['accuracy'] for r in top_results]
    errors = [r['std'] for r in top_results]
    
    # Reverse order for better visualization
    names = names[::-1]
    accuracies = accuracies[::-1]
    errors = errors[::-1]
    
    plt.barh(range(len(names)), accuracies, xerr=errors, align='center', alpha=0.7)
    plt.yticks(range(len(names)), names)
    plt.xlabel('Accuracy')
    plt.title('Top 10 Pipeline Comparison - Accuracy')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    
    # Highlight the best pipeline
    best_idx = names.index(best_pipeline_name) if best_pipeline_name in names else -1
    if best_idx >= 0:
        plt.barh(best_idx, accuracies[best_idx], xerr=errors[best_idx], align='center', alpha=0.7, color='green')
    
    plt.tight_layout()
    plt.savefig(os.path.join(models_dir, 'advanced_pipeline_comparison.png'))
    
    return best_pipeline, pipeline_info

def main():
    """
    Main function to find the best pipeline with enhanced accuracy
    """
    logger.info("Starting Advanced EEG Pipeline Optimizer")
    
    try:
        # Load preprocessed data
        X, y, preprocessing_info = load_data()
        
        # Try data augmentation if class imbalance exists
        class_counts = np.bincount(y)
        if len(class_counts) > 1 and max(class_counts) / min(class_counts) > 1.5:
            logger.info("Class imbalance detected, trying data augmentation...")
            X_aug, y_aug = augment_data(X, y)
            if len(X_aug) > len(X):
                logger.info("Using augmented dataset for pipeline evaluation")
                X, y = X_aug, y_aug
        
        # Find the best pipeline
        best_pipeline, pipeline_info = evaluate_and_find_best_pipeline(X, y, scoring='accuracy')
        
        if best_pipeline is not None:
            logger.info(f"Successfully found and saved advanced pipeline!")
            logger.info(f"Use this pipeline for prediction by loading from: {pipeline_info['pipeline_path']}")
        else:
            logger.error("Failed to find best pipeline")
            
    except Exception as e:
        logger.error(f"Error in advanced pipeline optimizer: {e}")
        raise e

if __name__ == "__main__":
    main()
